In [1]:
import pandas as pd
import gc
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error

In [2]:
calendar = pd.read_csv("C:/Users/shrut/Desktop/walmart/calendar.csv")

In [3]:
calendar['is_holiday'] = calendar['event_name_1'].notna().astype('int8')
calendar_clean = calendar[['date', 'd', 'wday', 'is_holiday', 'wm_yr_wk','snap_CA','snap_TX','snap_WI']]
calendar_clean['is_weekend']=calendar_clean['wday'].isin([1,2]).astype(int)
date_to_weekend_map=dict(zip(calendar_clean['d'],calendar_clean['is_weekend']))
calendar_clean.head(1)
output='C:/Users/shrut/Desktop/walmart/clean_calender.csv'
calendar_clean.to_csv(output, index=False)


C:\Users\shrut\AppData\Local\Temp\ipykernel_19164\1962952173.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  calendar_clean['is_weekend']=calendar_clean['wday'].isin([1,2]).astype(int)


In [4]:
prices = pd.read_csv("C:/Users/shrut/Desktop/walmart/sell_prices.csv")

In [5]:
sales = pd.read_csv("C:/Users/shrut/Desktop/walmart/sales_train_validation.csv")

In [6]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
le = LabelEncoder()

for col in cat_cols:
    sales[col] = le.fit_transform(sales[col].astype(str))

sales.head(1)

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,14370,1437,3,1,0,0,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1


In [ ]:
df = sales
day_cols = [c for c in df.columns if c.startswith("d_")]
values = df[day_cols].values
n_days = len(day_cols)
chunk_size = 2000
output_filename = "C:/Users/shrut/Desktop/walmart/walmart_m5_melted_data.csv"

if os.path.exists(output_filename):
    os.remove(output_filename)

for start in range(0, len(df), chunk_size):
    end = start + chunk_size
    
    chunk_values = values[start:end]
    n_series = chunk_values.shape[0]
    long_values = chunk_values.reshape(-1)
    
    chunk_df = pd.DataFrame({
        "id": np.repeat(df["id"].values[start:end], n_days),
        "item_id": np.repeat(df["item_id"].values[start:end], n_days),
        "dept_id": np.repeat(df["dept_id"].values[start:end], n_days),
        "cat_id": np.repeat(df["cat_id"].values[start:end], n_days),
        "store_id": np.repeat(df["store_id"].values[start:end], n_days),
        "state_id": np.repeat(df["state_id"].values[start:end], n_days),
        "day": np.tile(day_cols, n_series),
        "sales": long_values})
    
    chunk_df['sales'] = chunk_df['sales'].astype('int16')
    
    if start == 0:
        chunk_df.to_csv(output_filename, mode='w', index=False)
    else:
        chunk_df.to_csv(output_filename, mode='a', index=False, header=False)
    
    del chunk_df
    gc.collect()

In [8]:
osales = pd.read_csv('C:/Users/shrut/Desktop/walmart/sales_train_validation.csv', usecols=['item_id', 'store_id'])

store_codes = sorted(osales['store_id'].unique())
store_map = {text: code for code, text in enumerate(store_codes)}

item_codes = sorted(osales['item_id'].unique())
item_map = {text: code for code, text in enumerate(item_codes)}
del osales
gc.collect()

prices['store_id'] = prices['store_id'].map(store_map)
prices['item_id'] = prices['item_id'].map(item_map)

prices.dropna(subset=['store_id', 'item_id'], inplace=True)
prices['store_id'] = prices['store_id'].astype('int64')
prices['item_id'] = prices['item_id'].astype('int64')

prices.head(1)

,store_id,item_id,wm_yr_wk,sell_price
0,0,1437,11325,9.58


In [9]:
sales_long = pd.read_csv('C:/Users/shrut/Desktop/walmart/walmart_m5_melted_data.csv')
sales_long['sales'] = sales_long['sales'].astype('int16')

In [10]:

# STEP 1: CALENDAR MAPS TAIYAR KARNA (DAILY LEVEL PAR)
calendar_clean['is_weekend'] = calendar_clean['wday'].isin([1, 2]).astype('int8')

# Kal ko lag aur rolling features ke liye asli 'date' column kaam aayega
date_map = dict(zip(calendar_clean['d'], calendar_clean['date']))
wmyrwk_map = dict(zip(calendar_clean['d'], calendar_clean['wm_yr_wk']))
weekend_map = dict(zip(calendar_clean['d'], calendar_clean['is_weekend']))

# Baki bache weekly features (holidays aur SNAP) ko daily track karne ke liye
calendar_clean['date_str'] = calendar_clean['date'] # temporary column for mapping

holiday_map = dict(zip(calendar_clean['d'], calendar_clean['is_holiday']))
snap_ca_map = dict(zip(calendar_clean['d'], calendar_clean['snap_CA']))
snap_tx_map = dict(zip(calendar_clean['d'], calendar_clean['snap_TX']))
snap_wi_map = dict(zip(calendar_clean['d'], calendar_clean['snap_WI']))

del calendar_clean
gc.collect()

# STEP 2: CHUNKING LOOP (SIRF MERGE AUR MAPPING KE LIYE)
csv_path = 'C:/Users/shrut/Desktop/walmart/walmart_m5_melted_data.csv'
chunk_size = 1_000_000
daily_chunks = []


for chunk in pd.read_csv(csv_path, chunksize=chunk_size, usecols=['item_id', 'store_id', 'day', 'sales']):
    
    # Memory downcasting (Bohat zaroori hai crash se bachne ke liye)
    chunk['sales'] = chunk['sales'].astype('int16')
    chunk['item_id'] = chunk['item_id'].astype('int16')
    chunk['store_id'] = chunk['store_id'].astype('int8')
    
    # Direct daily mappings (No Aggregation)
    chunk['date'] = chunk['day'].map(date_map)
    chunk['wm_yr_wk'] = chunk['day'].map(wmyrwk_map)
    chunk['is_weekend'] = chunk['day'].map(weekend_map).astype('int8')
    chunk['is_holiday'] = chunk['day'].map(holiday_map).astype('int8')
    chunk['snap_CA'] = chunk['day'].map(snap_ca_map).astype('int8')
    chunk['snap_TX'] = chunk['day'].map(snap_tx_map).astype('int8')
    chunk['snap_WI'] = chunk['day'].map(snap_wi_map).astype('int8')
    
    # Hum 'day' (`d_1`, `d_2`) column ko drop kar rahe hain kyunki 'date' aa chuka hai
    chunk.drop(columns=['day'], inplace=True)
    
    daily_chunks.append(chunk)

# Saare processed chunks ko jodna
df_final = pd.concat(daily_chunks, ignore_index=True)

del daily_chunks, date_map, wmyrwk_map, weekend_map, holiday_map, snap_ca_map, snap_tx_map, snap_wi_map
gc.collect()


# STEP 3: PRICES CO MERGE KARNA
prices['item_id'] = prices['item_id'].astype('int16')
prices['store_id'] = prices['store_id'].astype('int8')
prices['sell_price'] = prices['sell_price'].astype('float32')

# Final table ke sath price table ka direct left merge
df_final = pd.merge(df_final, prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')

# Fuzool meta column drop karna aur missing prices ko handle karna
df_final.drop(columns=['wm_yr_wk'], inplace=True)
df_final['sell_price'] = df_final['sell_price'].fillna(0.0)

del prices
gc.collect()

# STEP 4: FINAL DATA CLEANING & SORTING
# Date column ko real datetime banana taaki training script mein mubaarbaad error na aaye
df_final['date'] = pd.to_datetime(df_final['date'])

# Chronological order mein save karna taaki lags sahi ban sakein
df_final = df_final.sort_values(by=['item_id', 'store_id', 'date']).reset_index(drop=True)

# Save to CSV
output_filename = "C:/Users/shrut/Desktop/walmart/walmart_daily_final.csv"
df_final.to_csv(output_filename, index=False)



C:\Users\shrut\AppData\Local\Temp\ipykernel_19164\1944136154.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  calendar_clean['is_weekend'] = calendar_clean['wday'].isin([1, 2]).astype('int8')
C:\Users\shrut\AppData\Local\Temp\ipykernel_19164\1944136154.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  calendar_clean['date_str'] = calendar_clean['date'] # temporary column for mapping


In [11]:
csv_path = 'C:/Users/shrut/Desktop/walmart/walmart_daily_final.csv'
chunk_size = 900_000
weekly_chunks = []


# 1. Loop ke andar se aggregation karna
for i, chunk in enumerate(pd.read_csv(csv_path, chunksize=chunk_size)):
    print(f"🔄 Processing Chunk {i+1}...")
    chunk['date']=pd.to_datetime(chunk['date'])
    chunk['date']=chunk['date'].dt.to_period('W').dt.start_time
    
    # Temporary weekend sales nikalna
    chunk['weekend_sales'] = chunk['sales'] * chunk['is_weekend']
    
    #  SNAP ACTIVE LOGIC: Store ke state ke hisab se 1 ya 0 set karna
    # M5 Dataset Mapping: Stores 0,1,2,3 -> CA | 4,5,6 -> TX | 7,8,9 -> WI
    cond_ca = chunk['store_id'].isin([0, 1, 2, 3])
    cond_tx = chunk['store_id'].isin([4, 5, 6])
    
    chunk['snap_active'] = np.select(
        [cond_ca, cond_tx], 
        [chunk['snap_CA'], chunk['snap_TX']], 
        default=chunk['snap_WI']
    ).astype('int8')
    
    # Chunk level compress (Weekly aggregation)
    chunk_weekly = chunk.groupby(['item_id', 'store_id', 'date'], as_index=False).agg({
        'sales': 'sum',
        'weekend_sales': 'sum',
        'is_holiday': 'max',      # Agar hafte me ek bhi din chutti thi
        'snap_active': 'max',      # Agar us hafte us store me snap active tha
        'sell_price': 'mean'       # Hafte ki average price
    })
    
    weekly_chunks.append(chunk_weekly)
    del chunk, chunk_weekly
    gc.collect()

weekly_df = pd.concat(weekly_chunks, ignore_index=True)
del weekly_chunks
gc.collect()

# Final combine (Cross-chunk duplicate weeks ko hatane ke liye)
df_weekly_final = weekly_df.groupby(['item_id', 'store_id', 'date'], as_index=False).agg({
    'sales': 'sum',
    'weekend_sales': 'sum',
    'is_holiday': 'max',
    'snap_active': 'max',
    'sell_price': 'mean'
}).copy()

del weekly_df
gc.collect()

# Sahi Percentage calculation
sales_vector = df_weekly_final['sales'].values
weekend_vector = df_weekly_final['weekend_sales'].values

df_weekly_final['weekend_sales_pct'] = np.where(
    sales_vector > 0,
    (weekend_vector / sales_vector) * 100.0,
    0.0
)

# Fuzool columns hatana
df_weekly_final.drop(columns=['weekend_sales'], inplace=True)

# Save the final neat csv
output_filename = "C:/Users/shrut/Desktop/walmart/walmart_weekly_perfect.csv"
df_weekly_final.to_csv(output_filename, index=False)


🔄 Processing Chunk 1...
🔄 Processing Chunk 2...
🔄 Processing Chunk 3...
🔄 Processing Chunk 4...
🔄 Processing Chunk 5...
🔄 Processing Chunk 6...
🔄 Processing Chunk 7...
🔄 Processing Chunk 8...
🔄 Processing Chunk 9...
🔄 Processing Chunk 10...
🔄 Processing Chunk 11...
🔄 Processing Chunk 12...
🔄 Processing Chunk 13...
🔄 Processing Chunk 14...
🔄 Processing Chunk 15...
🔄 Processing Chunk 16...
🔄 Processing Chunk 17...
🔄 Processing Chunk 18...
🔄 Processing Chunk 19...
🔄 Processing Chunk 20...
🔄 Processing Chunk 21...
🔄 Processing Chunk 22...
🔄 Processing Chunk 23...
🔄 Processing Chunk 24...
🔄 Processing Chunk 25...
🔄 Processing Chunk 26...
🔄 Processing Chunk 27...
🔄 Processing Chunk 28...
🔄 Processing Chunk 29...
🔄 Processing Chunk 30...
🔄 Processing Chunk 31...
🔄 Processing Chunk 32...
🔄 Processing Chunk 33...
🔄 Processing Chunk 34...
🔄 Processing Chunk 35...
🔄 Processing Chunk 36...
🔄 Processing Chunk 37...
🔄 Processing Chunk 38...
🔄 Processing Chunk 39...
🔄 Processing Chunk 40...
🔄 Process

C:\Users\shrut\AppData\Local\Temp\ipykernel_19164\994711420.py:61: RuntimeWarning: invalid value encountered in divide
  (weekend_vector / sales_vector) * 100.0,


In [12]:
df_weekly_final=pd.read_csv("C:/Users/shrut/Desktop/walmart/walmart_weekly_perfect.csv")

In [13]:
df_weekly_final.head(1)

,item_id,store_id,date,sales,is_holiday,snap_active,sell_price,weekend_sales_pct
0,0,0,2011-01-24,3,0,0,2.0,100.0
